# Jawaban Latihan 1
- Load data 05
- Audio Augmentation 40
- Menu 05
- Predict 15
- Export File 20
- Logic 15

# Step 1: Import Libray

In [10]:
import torch
import torchaudio
import librosa
import soundfile as sf
import numpy as np
from pydub import AudioSegment
import random

# Step 2: Settings Wav2Vec Pre Train Model and Audio File

In [11]:
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

In [12]:
AUDIO_FILE = "speech.wav"

# Step 3: Predict Function / Wav2Vec

In [13]:
# Predict Text Function

def predict_text(AUDIO_FILE):
    waveform, sr = torchaudio.load(AUDIO_FILE)
    
    if sr != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate)
    
    with torch.inference_mode():
        emission, _ = model(waveform)
        
    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0
            
        def forward(self, emission: torch.Tensor)->str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return "".join([self.labels[i] for i in indices])
        
    decoder = GreedyCTCDecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript
            

# Step 4: Audio Augmentation + Export File

In [14]:
# validasi nama
def validate_filename(filename):
    if filename.endswith(".wav"):
        return True
    else:
        print("Filename must ends with .wav")
        return False

In [15]:
# augmentation
def adjust_volume(audio, db):
    return audio + db

def adjust_pitch(audio_path, pitch_rate):
    data, sample_rate = librosa.load(audio_path, sr=None)
    shifted_audio = librosa.effects.pitch_shift(data, sr=sample_rate, n_steps=pitch_rate)
    temp_file = "shifted.wav"
    sf.write(temp_file, shifted_audio, sample_rate)
    new_audio = AudioSegment.from_wav(temp_file)
    return new_audio

def apply_fade_in(audio, duration):
    return audio.fade_in(duration)

def apply_fade_out(audio, duration):
    return audio.fade_out(duration)

def export_file(audio, file_name):
    audio.export(file_name, format="wav")

def export_function(audio):
    while True:
        filename = input("Filename: ")
        if filename.endswith(".wav"):
            export_file(audio, filename)
            break
        else:
            print("Filename must ends with .wav")


# Step 5: Menu 1 (aug_menu())

In [16]:
def menu1():
    audio = AudioSegment.from_wav(AUDIO_FILE)
    while True:
        print("1. Increase or Decrease Volume")
        print("2. Increase or Decrease Pitch")
        print("3. Insert Fade in Effect")
        print("4. Insert Fade Out Effect")
        print("5. Back to main Menu")

        choice = int(input("Select Menu [1-5]: "))
        if choice == 1:
            while True:
                db_input = int(input("Input db: "))
                if db_input > -10000 and db_input < 100:
                    break
                else:
                    print("db must be between -10000 and 100")
            audio = adjust_volume(audio, db_input)
            export_function(audio)

        elif choice == 2:
            while True:
                pitch = int(input("Input Pitch: "))
                if pitch > -10 and pitch < 10:
                    break
                else:
                    print("Pitch must be between -10 and 10")
            audio = adjust_pitch(AUDIO_FILE, pitch)
            export_function(audio)

        elif choice == 3:
            duration = random.randint(1, 5) * 1000
            audio = apply_fade_in(audio, duration)
            export_function(audio)

        elif choice == 4:
            duration = random.randint(1, 5) * 1000
            audio = apply_fade_out(audio, duration)
            export_function(audio)

        elif choice == 5:
            break

        else:
            print("Invalid input must be between 1-5")

In [17]:
def mainmenu():
    while True:
        print("1. Audio Augmentation")
        print("2. Predict Text")
        print("3. Exit")
        
        choice = input(">> ")
        if choice == "1":
            print("Audio Augmentation Menu")
            menu1()
        elif choice == "2":
            transcript = predict_text(AUDIO_FILE)
            print(f"Hasil Transkrip: {transcript}")
        elif choice == "3":
            break
        else:
            print("Invalid Input")
    print("Main Menu Closed")

mainmenu()

1. Audio Augmentation
2. Predict Text
3. Exit
Invalid Input
1. Audio Augmentation
2. Predict Text
3. Exit
Audio Augmentation Menu
1. Increase or Decrease Volume
2. Increase or Decrease Pitch
3. Insert Fade in Effect
4. Insert Fade Out Effect
5. Back to main Menu
Filename must ends with .wav
Filename must ends with .wav
Filename must ends with .wav
Filename must ends with .wav
Filename must ends with .wav
Filename must ends with .wav
Filename must ends with .wav
Filename must ends with .wav
Filename must ends with .wav
Filename must ends with .wav
Filename must ends with .wav
Filename must ends with .wav


KeyboardInterrupt: Interrupted by user